# Packages

In [1]:
import pickle

import pandas as pd

# Loading Dependencies

## Datasets

In [2]:
df_events = pd.read_csv("../data/events.csv")
df_products = pd.read_csv("../data/products.csv")

## Scaler

In [3]:
with open("../model/model.pkl", "rb") as f:
    artifact = pickle.load(f)

model = artifact["model"]          # sklearn LogisticRegression
scaler = artifact["scaler"]        # sklearn StandardScaler
feature_cols = artifact["feature_cols"]  # ordem exata das features esperadas

# Helper Functions

In [5]:
def compute_user_top_affinity_category(
    events: pd.DataFrame,
    products: pd.DataFrame,
) -> pd.DataFrame:
    """
    For each user, the top-affinity category is the one with the most
    event rows (after joining events with products by product_id).
    Tie-break: highest count, then category name alphabetically.
    """
    events_with_category = events.merge(
        products[["product_id", "category"]],
        on="product_id",
        how="inner",
    )
    category_counts = (
        events_with_category.groupby(["user_id", "category"], as_index=False)
        .size()
        .rename(columns={"size": "interaction_count"})
    )
    top_affinity = (
        category_counts.sort_values(
            ["user_id", "interaction_count", "category"],
            ascending=[True, False, True],
        )
        .drop_duplicates("user_id")
        .rename(columns={"category": "top_affinity_category"})[
            ["user_id", "top_affinity_category"]
        ]
    )
    return top_affinity


In [ ]:
def build_features(
    events: pd.DataFrame,
    products: pd.DataFrame,
    feature_cols: list[str] | None = None,
    pairs: pd.DataFrame | None = None
) -> pd.DataFrame:
    """
    Build model features for user-product pairs.
    Parameters
    ----------
    events:
        events.csv
    products:
        products.csv
    pairs:
        Optional DataFrame with columns [user_id, product_id].
        If None, uses every (user_id, product_id) seen in events.
    """
    if pairs is None:
        pairs = events[["user_id", "product_id"]].drop_duplicates()
    # 1) interactions: historical count per user-product
    interactions = (
        events.groupby(["user_id", "product_id"], as_index=False)
        .size()
        .rename(columns={"size": "interactions"})
    )
    # 2-4) product features from products.csv
    product_features = products[
        ["product_id", "category", "price", "avg_rating", "popularity_score"]
    ]
    # 5) user affinity match
    top_affinity = compute_user_top_affinity_category(events, products)
    features = (
        pairs.merge(interactions, on=["user_id", "product_id"], how="left")
        .merge(product_features, on="product_id", how="left")
        .merge(top_affinity, on="user_id", how="left")
    )
    features["interactions"] = features["interactions"].fillna(0).astype(int)
    features["user_affinity_match"] = (
        features["category"] == features["top_affinity_category"]
    ).astype("Int64")  # nullable int for cold-start users
    return features[
        ["user_id", "product_id", *feature_cols]
    ]

In [7]:
def build_features_for_user(
    user_id: str,
    events: pd.DataFrame,
    products: pd.DataFrame,
    candidate_product_ids: list[str] | None = None,
) -> pd.DataFrame:
    """
    Build features for one user against candidate products.
    Useful for the recommendation endpoint.
    """
    if candidate_product_ids is None:
        candidate_product_ids = products["product_id"].tolist()
    pairs = pd.DataFrame(
        {
            "user_id": user_id,
            "product_id": candidate_product_ids,
        }
    )
    return build_features(events, products, pairs=pairs)

In [10]:
def scale_features(
    features: pd.DataFrame,
    scaler,
    feature_cols: list[str],
) -> pd.DataFrame:
    """
    Scale raw features using the fitted StandardScaler from model.pkl.

    Important:
    - Columns must follow the exact order stored in feature_cols.
    - Use transform(), not fit_transform(), because the scaler was fit offline.
    - Missing user_affinity_match (cold start) is filled with 0 before scaling.
    """
    X = features[feature_cols].astype(float).fillna(0).to_numpy()
    X_scaled = scaler.transform(X)

    return pd.DataFrame(
        X_scaled,
        columns=feature_cols,
        index=features.index,
    )

# Derivating Feature Variables

In [4]:
FEATURE_COLS = [
    "interactions",
    "price",
    "avg_rating",
    "popularity_score",
    "user_affinity_match",
]

In [8]:
df = build_features(df_events, df_products)

In [9]:
df

,user_id,product_id,interactions,price,avg_rating,popularity_score,user_affinity_match
0,u_0231,p_042,1,278.10,4.0,0.068,0
1,u_0078,p_012,6,752.81,4.9,0.096,1
2,u_0322,p_059,2,104.16,4.5,0.386,1
3,u_0121,p_045,4,273.64,3.1,0.327,1
4,u_0316,p_036,1,621.59,4.0,0.262,1
...,...,...,...,...,...,...,...
5281,u_0084,p_030,1,622.35,3.8,0.801,1
5282,u_0063,p_057,1,353.48,4.5,0.326,0
5283,u_0162,p_044,1,262.57,3.6,0.127,1
5284,u_0363,p_025,1,443.30,3.9,0.323,0


# Scaling Features

In [11]:
df_scaled = scale_features(df, scaler, feature_cols)

df_model_input = pd.concat(
    [df[["user_id", "product_id"]], df_scaled],
    axis=1,
)

df_model_input.head()

,user_id,product_id,interactions,price,avg_rating,popularity_score,user_affinity_match
0,u_0231,p_042,-0.594659,-0.513768,0.013679,-1.250936,-1.351569
1,u_0078,p_012,5.196365,1.539539,1.520086,-1.071331,0.739881
2,u_0322,p_059,0.563546,-1.266127,0.850572,0.788868,0.739881
3,u_0121,p_045,2.879955,-0.533060,-1.492728,0.410414,0.739881
4,u_0316,p_036,-0.594659,0.971961,0.013679,-0.006527,0.739881


# Testing Model

In [12]:
# Optional: run inference with scaled features
df["purchase_score"] = model.predict_proba(df_scaled[feature_cols].to_numpy())[:, 1]

df.sort_values("purchase_score", ascending=False).head()

,user_id,product_id,interactions,price,avg_rating,popularity_score,user_affinity_match,purchase_score
987,u_0287,p_053,8,457.80,4.0,0.234,1,0.673850
3158,u_0087,p_040,8,693.22,4.9,0.541,1,0.646711
414,u_0046,p_043,7,69.58,3.6,0.047,1,0.608823
3007,u_0491,p_027,7,645.71,4.7,0.114,1,0.572150
911,u_0330,p_044,6,262.57,3.6,0.127,1,0.479540
